# Custom Dataset Workflow

This notebook shows how to create a runtime dataset with `custom_dataset(...)`, load prices without editing `configs/datasets.toml`, build features, fit a very small linear baseline, run the shared backtest, write artifacts, and log the run to MLflow.

## Running This Notebook In Colab

If you are in Colab, clone the repo and install the package first. If you are local, the fallback path below keeps `repo_root` pointed at the current checkout.

In [1]:
from pathlib import Path
import os
import sys

if 'google.colab' in sys.modules:
    repo_url = 'https://github.com/adamthorne27/Portfolio-Optimization-Lib.git'
    repo_root = Path('/content/Portfolio-Optimization-Lib')
    if not repo_root.exists():
        !git clone {repo_url}
    %pip install -q -e /content/Portfolio-Optimization-Lib
else:
    repo_root = Path('../../').resolve()

repo_root

PosixPath('/Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer')

In [2]:
import numpy as np
import pandas as pd

from portfolio_toolkit import (
    backtest_predictions,
    build_features,
    custom_dataset,
    load_prices,
    log_backtest,
    log_predictions,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    write_backtest_artifacts,
)


/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Construct A Runtime Dataset

This creates a first-class `DatasetSpec` in memory. It behaves like a normal shared preset everywhere else in the toolkit, but it does not require any TOML edits.

In [3]:
dataset = custom_dataset(
    tickers=['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL'],
    start='2018-01-02',
    end='2025-12-31',
    benchmark='SPY',
    name='custom_big_tech_demo',
)

dataset

DatasetSpec(name='custom_big_tech_demo', tickers=['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL'], benchmark_ticker='SPY', start_date=datetime.date(2018, 1, 2), end_date=datetime.date(2025, 12, 31), train_start=datetime.date(2018, 1, 2), train_end=datetime.date(2022, 10, 19), val_start=datetime.date(2022, 10, 20), val_end=datetime.date(2024, 5, 25), test_start=datetime.date(2024, 5, 26), test_end=datetime.date(2025, 12, 31), cost_bps=10.0, default_benchmark='SPY', kind='custom', dataset_id='custom_big_tech_demo')

In [4]:
split_dates(dataset)

{'train': (Timestamp('2018-01-02 00:00:00'), Timestamp('2022-10-19 00:00:00')),
 'val': (Timestamp('2022-10-20 00:00:00'), Timestamp('2024-05-25 00:00:00')),
 'test': (Timestamp('2024-05-26 00:00:00'), Timestamp('2025-12-31 00:00:00'))}

## 2. Load Prices And Slice Splits

The price loader now accepts a runtime `DatasetSpec`, caches it using a deterministic custom dataset identifier, and validates the benchmark and ticker universe the same way it does for preset datasets.

In [5]:
prices = load_prices(dataset, repo_root=repo_root)
train_prices = slice_split(prices, dataset, 'train')
val_prices = slice_split(prices, dataset, 'val')
test_prices = slice_split(prices, dataset, 'test')

print('Full price frame:', prices.shape)
print('Train price frame:', train_prices.shape)
print('Val price frame:', val_prices.shape)
print('Test price frame:', test_prices.shape)
prices.head()

Full price frame: (12066, 8)
Train price frame: (7254, 8)
Val price frame: (2406, 8)
Test price frame: (2406, 8)


,date,ticker,open,high,low,close,adj_close,volume
0,2018-01-02,AAPL,42.540001,43.075001,42.314999,43.064999,40.304173,102223600
1,2018-01-03,AAPL,43.132500,43.637501,42.990002,43.057499,40.297161,118071600
2,2018-01-04,AAPL,43.134998,43.367500,43.020000,43.257500,40.484341,89738400
3,2018-01-05,AAPL,43.360001,43.842499,43.262501,43.750000,40.945251,94640000
4,2018-01-08,AAPL,43.587502,43.902500,43.482498,43.587502,40.793179,82271200


## 3. Build Shared Features And A Simple Target

Use a small built-in feature set and a forward return target. Then fit a minimal linear baseline with `numpy.linalg.lstsq` so the focus stays on the dataset/runtime flow rather than model complexity.

In [ ]:
feature_names = ['momentum_20d', 'vol_20d', 'relative_momentum_20d_vs_spy']
features = build_features(prices, feature_names=feature_names)
target = make_forward_return_target(prices, 5)

model_frame = features.merge(target, on=['date', 'ticker'], how='inner')
model_frame = model_frame.dropna(subset=feature_names + ['forward_return_5d']).reset_index(drop=True)

train_frame = slice_split(model_frame, dataset, 'train')
test_frame = slice_split(model_frame, dataset, 'test')

X_train = train_frame[feature_names].to_numpy(dtype=float)
y_train = train_frame['forward_return_5d'].to_numpy(dtype=float)
X_test = test_frame[feature_names].to_numpy(dtype=float)

X_train_design = np.column_stack([np.ones(len(X_train)), X_train])
coefficients, *_ = np.linalg.lstsq(X_train_design, y_train, rcond=None)

predicted_returns = np.column_stack([np.ones(len(X_test)), X_test]) @ coefficients
predictions = test_frame.loc[:, ['date', 'ticker']].copy()
predictions['horizon'] = 5
predictions['expected_return'] = predicted_returns
predictions = validate_prediction_frame(predictions, dataset_name=dataset, horizon=5, repo_root=repo_root)
predictions.head()

## 4. Backtest And Write Standard Artifacts

The custom dataset now works with the normal backtest and reporting stack exactly like a preset dataset.

In [ ]:
result = backtest_predictions(dataset, predictions, builder='top_k_equal', k=3, repo_root=repo_root)
artifact_dir = repo_root / 'runs' / dataset.name / 'custom_dataset_demo'
artifact_paths = write_backtest_artifacts(result, artifact_dir)

pd.DataFrame([{'metric': key, 'value': value} for key, value in sorted(result.metrics.items())])

## 5. Log The Run To MLflow

When `start_run(...)` receives a runtime dataset, it uses the dataset identifier in the experiment name and logs a `dataset_spec.json` artifact automatically.

In [ ]:
with start_run('custom_dataset_demo', dataset, repo_root=repo_root):
    log_predictions(predictions)
    log_backtest(result)

artifact_paths